# 00 — Features: TF‑IDF + FinBERT + MacroTone
Ce notebook construit des **features documentaires** à partir de phrases :  
- TF‑IDF (1–2‑grammes) → agrégation par `Doc_ID` → **SVD** (LSA) + scree plot  
- **FinBERT** (optionnel) sur phrases → agrégation doc → histogramme de ton  
- Fusion → **MacroTone_index** = z(SVD1) + z(FinBERT_tone) + tracé temporel

> Ajuste les chemins ci‑dessous (CSV d’entrée et répertoire de sortie).

In [4]:
# Configuration des chemins (à adapter)
IN_CSV = r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\pre_processed_ECB_final.csv"   # <-- remplace par ton chemin
OUT_DIR = r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\analysis\out"                                        # répertoire de sortie
USE_FINBERT = True                                       # mets False si tu n'as pas transformers/Internet
FINBERT_ON_CLEAN = False                                 # True = scorer sur Parsed_Text_clean


In [5]:
# Imports
import os, math, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

def zscore(x):
    x = pd.Series(x, dtype='float64')
    return (x - x.mean()) / (x.std(ddof=0) + 1e-12)

In [6]:
from pathlib import Path
import pandas as pd

IN = Path(IN_CSV)                       # ex: .../dataset/pre_processed_ECB_final.csv
OUT_DIR = IN.parent / "outputs"         # dossier à côté du CSV
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Lire
df = pd.read_csv(IN)

# Drop colonnes "Unnamed" et vérifier les colonnes requises
drop_unnamed = [c for c in df.columns if str(c).lower().startswith("unnamed")]
df = df.drop(columns=drop_unnamed, errors="ignore")

required = ["Doc_ID", "Parsed_Text_clean", "Sentence_Rank"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Colonnes manquantes: {missing}")

# Trier
df = df.sort_values(["Doc_ID", "Sentence_Rank"]).reset_index(drop=True)

# Sauvegarder (sans index)
OUT_FILE = OUT_DIR / f"{IN.stem}__sorted.csv"
df.to_csv(OUT_FILE, index=False)
print(f"[OK] Écrit -> {OUT_FILE}")
from pathlib import Path
import pandas as pd

IN = Path(IN_CSV).resolve()
OUT_DIR = IN.parent / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sanity checks
assert IN.exists() and IN.is_file(), f"Introuvable: {IN}"

try:
    df = pd.read_csv(IN)
except PermissionError as e:
    raise SystemExit(
        f"PermissionError: {IN}\n"
        "→ Ferme Excel/VS Code/Notebook qui utilisent ce fichier, "
        "ou relance le kernel si un script l'écrivait encore."
    ) from e

# Garder uniquement les colonnes utiles et supprimer Parsed_Text
cols_keep = [c for c in df.columns if c != "Parsed_Text"]
if "Parsed_Text_clean" not in cols_keep:
    raise ValueError("Colonne 'Parsed_Text_clean' manquante dans le CSV source.")
df = df[cols_keep]

# (optionnel) tri si dispo
for key in ["Doc_ID", "Sentence_Rank"]:
    if key not in df.columns:
        break
else:
    df = df.sort_values(["Doc_ID","Sentence_Rank"]).reset_index(drop=True)

# Sauvegarde
OUT_FILE = OUT_DIR / f"{IN.stem}__no_parsed_text.csv"
df.to_csv(OUT_FILE, index=False)
print(f"[OK] Écrit -> {OUT_FILE}")


[OK] Écrit -> C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\outputs\pre_processed_ECB_final__sorted.csv
[OK] Écrit -> C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\outputs\pre_processed_ECB_final__no_parsed_text.csv


## TF‑IDF (1–2‑grammes) → Agrégation Doc → SVD
On vectorise les **phrases**, puis on **moyenne** par `Doc_ID` pour avoir un vecteur **documentaire**.  
Réduction de dimension par **TruncatedSVD** et sauvegarde des composantes.

In [ ]:
# TF-IDF sur phrases
texts = df['Parsed_Text_clean'].fillna('').astype(str).tolist()
n_sent = len(texts)
min_df = 3 if n_sent > 100 else 1
max_df = 0.95 if n_sent > 100 else 1.0

tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=min_df, max_df=max_df)
X_sent = tfidf.fit_transform(texts)

# Agrégation doc (moyenne des phrases par Doc_ID)
doc_codes = df['Doc_ID'].astype('category')
doc_index = doc_codes.cat.codes.values
n_docs = doc_codes.cat.categories.size

row_ids = doc_index
col_ids = np.arange(len(row_ids))
G = sparse.csr_matrix((np.ones_like(col_ids), (row_ids, col_ids)), shape=(n_docs, len(row_ids)))
doc_counts = np.asarray(G.sum(axis=1)).ravel()
X_doc = (G @ X_sent)
doc_counts_safe = np.maximum(doc_counts, 1)[:,None]
X_doc = X_doc.multiply(1.0 / doc_counts_safe)

# SVD (LSA)
svd_k = min(100, max(2, int(min(X_doc.shape) * 0.6))) if min(X_doc.shape) > 2 else 2
svd = TruncatedSVD(n_components=svd_k, random_state=42)
X_doc_svd = svd.fit_transform(X_doc)

# Sauvegarde
tfidf_svd_cols = [f'tfidf_svd_{i+1}' for i in range(X_doc_svd.shape[1])]
df_tfidf_doc = pd.DataFrame(X_doc_svd, columns=tfidf_svd_cols)
df_tfidf_doc.insert(0, 'Doc_ID', doc_codes.cat.categories.values)
df_tfidf_doc.to_csv(OUT / 'ecb_features_tfidf_svd.csv', index=False)

# Scree plot (variance expliquée cumulée)
plt.figure()
cumvar = np.cumsum(svd.explained_variance_ratio_)
plt.plot(np.arange(1, len(cumvar)+1), cumvar, marker='o')
plt.xlabel('Number of SVD components')
plt.ylabel('Cumulative explained variance')
plt.title('TF-IDF → TruncatedSVD (document-level)')
plt.tight_layout()
plt.savefig(OUT / 'tfidf_svd_scree.png')
plt.show()

df_tfidf_doc.head()

## FinBERT (optionnel)
Score **pos/neu/neg** par phrase, agrégation **par Doc_ID**, puis `finbert_tone = pos − neg`.  
Histogramme du ton documentaire et export CSVs.

In [ ]:
# FinBERT (optionnel)
if USE_FINBERT:
    try:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline
        model_name = 'ProsusAI/finbert'
        tok = AutoTokenizer.from_pretrained(model_name)
        mdl = AutoModelForSequenceClassification.from_pretrained(model_name)
        pipe = TextClassificationPipeline(model=mdl, tokenizer=tok, return_all_scores=True, truncation=True)
        finbert_ok = True
    except Exception as e:
        print('[WARN] FinBERT non disponible:', e)
        finbert_ok = False
else:
    finbert_ok = False

if finbert_ok:
    fin_text_col = 'Parsed_Text_clean' if FINBERT_ON_CLEAN or 'Parsed_Text' not in df.columns else 'Parsed_Text'
    texts_raw = df[fin_text_col].fillna('').astype(str).tolist()

    scores = []
    B = 64
    for i in range(0, len(texts_raw), B):
        out = pipe(texts_raw[i:i+B])
        scores.extend(out)

    def to_vec(score_list):
        d = {s['label'].lower(): s['score'] for s in score_list}
        return (d.get('positive',0.0), d.get('neutral',0.0), d.get('negative',0.0))

    mat = np.array([to_vec(s) for s in scores], dtype='float64')
    sent_df = pd.DataFrame(mat, columns=['finbert_pos','finbert_neu','finbert_neg'])
    sent_df.insert(0, 'Doc_ID', df['Doc_ID'].values)
    sent_df.insert(1, 'Sentence_Rank', df['Sentence_Rank'].values)
    sent_df.to_csv(OUT / 'ecb_finbert_sentence_scores.csv', index=False)

    doc_sent = sent_df.groupby('Doc_ID')[['finbert_pos','finbert_neu','finbert_neg']].mean().reset_index()
    doc_sent['finbert_tone'] = doc_sent['finbert_pos'] - doc_sent['finbert_neg']
    doc_sent.to_csv(OUT / 'ecb_finbert_doc_scores.csv', index=False)

    # Histogramme du ton
    plt.figure()
    plt.hist(doc_sent['finbert_tone'].values, bins=15)
    plt.xlabel('FinBERT tone (pos - neg)')
    plt.ylabel('Count of documents')
    plt.title('Distribution of FinBERT tone (document-level)')
    plt.tight_layout()
    plt.savefig(OUT / 'finbert_tone_hist.png')
    plt.show()
else:
    doc_sent = pd.DataFrame({'Doc_ID': df_tfidf_doc['Doc_ID']})

doc_sent.head()

## Fusion + MacroTone + Série temporelle
On fusionne TF‑IDF‑SVD avec FinBERT (si dispo) et on calcule **MacroTone_index**.  
On trace la série temporelle si `Date` est disponible.

In [ ]:
# Fusion + MacroTone
feat = df_tfidf_doc.merge(doc_sent, on='Doc_ID', how='left')
if 'tfidf_svd_1' in feat.columns:
    feat['tfidf_svd_1_z'] = zscore(feat['tfidf_svd_1'])

if 'finbert_tone' in feat.columns:
    feat['finbert_tone_z'] = zscore(feat['finbert_tone'].fillna(0.0))
    feat['MacroTone_index'] = feat['tfidf_svd_1_z'].fillna(0.0) + feat['finbert_tone_z'].fillna(0.0)
else:
    feat['MacroTone_index'] = feat.get('tfidf_svd_1_z', pd.Series(0, index=feat.index))

# Attacher Date si dispo (pour plotting)
if 'Date' in df.columns:
    dates = df.drop_duplicates('Doc_ID')[['Doc_ID','Date']]
    feat = feat.merge(dates, on='Doc_ID', how='left')

feat.to_csv(OUT / 'ecb_features_combined.csv', index=False)
feat.head(10)

In [ ]:
# Série temporelle MacroTone (si Date existe)
if 'Date' in feat.columns:
    try:
        tmp = feat.drop_duplicates('Doc_ID').copy()
        tmp['Date_parsed'] = pd.to_datetime(tmp['Date'])
        tmp = tmp.sort_values('Date_parsed')
        plt.figure()
        plt.plot(tmp['Date_parsed'], tmp['MacroTone_index'], marker='o')
        plt.xlabel('Date')
        plt.ylabel('MacroTone index')
        plt.title('Composite macro-communication index over time')
        plt.tight_layout()
        plt.savefig(OUT / 'macrotone_timeseries.png')
        plt.show()
    except Exception as e:
        print('[WARN] Impossible de tracer la série temporelle:', e)